In [1]:
import sys; print(sys.executable)

c:\Users\anton\OneDrive\Desktop\internship\QuantifyingSchmidtNumber\.venv\Scripts\python.exe


In [2]:
%config InteractiveShell.cache_size = 0
from SDP_utils.isotopic_proj import isotypic_projector
import numpy as np
from toqito.matrix_ops import tensor
from tqdm import tqdm

lam = [2,1]
n = 3
k = sum(lam)
pi_lam = isotypic_projector(lam, n, k)

In [3]:
def interleave_perm(k: int, sys: int):
    p1 = np.array([i * sys for i in range(k)])
    res = p1
    for j in range(1, sys):
        res = np.append(res, p1 + j)
    return res


interleave_perm(4, 3)

array([ 0,  3,  6,  9,  1,  4,  7, 10,  2,  5,  8, 11])

Testing the diff between projected norm with $\Pi_\lambda \otimes I$ and $\Pi_\lambda \otimes \Pi_\lambda$

In [4]:
from toqito.rand import random_state_vector
from toqito.perms import permute_systems

state = random_state_vector(n**2)
stateCopies = tensor([state] * k)
stateCopies = permute_systems(stateCopies, interleave_perm(k, 2), dim=[n] * 6)

In [5]:
pi_I = tensor(pi_lam, np.identity(n**k))
pi_pi = tensor(pi_lam, pi_lam)

print(np.linalg.vector_norm(pi_I @ stateCopies))
print(np.linalg.vector_norm(pi_pi @ stateCopies))

print(np.allclose(pi_I @ stateCopies, pi_pi @ stateCopies))

0.5603105369041246
0.5603105369041246
True


we got the same result

$||\Pi_\lambda \otimes I \otimes I |\psi\rangle||^2$ and $||\Pi_\lambda \otimes \Pi_\lambda \otimes \Pi_\lambda |\psi\rangle||^2$

In [6]:
from toqito.rand import random_state_vector
from toqito.perms import permute_systems

state = random_state_vector(n**3)
stateCopies = tensor([state] * k)
stateCopies = permute_systems(stateCopies, interleave_perm(k, 3), dim=[n] * 3 * k)

In [7]:
# pi_I = tensor(pi_lam, np.identity(n**k),np.identity(n**k))
# print(np.linalg.vector_norm(pi_I @ stateCopies))
# pi_I = 0

# pi_pi = tensor(pi_lam, pi_lam, np.identity(n**k))
# print(np.linalg.vector_norm(pi_pi @ stateCopies))
# pi_pi = 0

# pi_pi_pi = tensor(pi_lam, pi_lam, pi_lam)
# print(np.linalg.vector_norm(pi_pi_pi @ stateCopies))
# pi_pi_pi = 0

We dont get the same result?

In [8]:
pi1 = isotypic_projector([2, 1], n, k)
pi2 = isotypic_projector([3], n, k)
pi3 = isotypic_projector([3], n, k)

pi_all = tensor(pi1, pi2, pi3)
print(pi_all @ stateCopies)

[0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]


# Test how orthogonal projectors act on Bipartite systems

In [9]:
n = 3
k = 2
state = random_state_vector(n**2)
stateCopies = tensor([state] * k)
stateCopies = permute_systems(stateCopies, interleave_perm(k, 2), dim=[n] * k * 2)

In [10]:
sym = isotypic_projector([2], n, k)
anti = isotypic_projector([1, 1], n, k)
sym_I = tensor(sym, np.identity(n**k))
I_anti = tensor(np.identity(n**k), anti)

print(np.linalg.norm(sym_I @ stateCopies))
print(np.linalg.norm(I_anti @ stateCopies))
print(np.linalg.norm(sym_I @ I_anti @ stateCopies))

0.9397533796248488
0.3418531636297591
1.78837496899204e-17


In [11]:
print(np.count_nonzero(~np.isclose(sym_I @ I_anti, 0)))

180


In [ ]:
def partitions(k: int, max_part=None):
    if max_part is None:
        max_part = k
    if k == 0:
        yield []
        return
    for first in range(min(k, max_part), 0, -1):
        for rest in partitions(k - first, first):
            yield [first] + rest


def unique_combinations(k: int, nb_vec: int):
    def unique_comb_rec(nb_remaining: int, start: int):
        if nb_remaining == 0:
            for i in range(start, k):
                yield (i,)
            return
        for i in range(start, k):
            for val in unique_comb_rec(nb_remaining - 1, i):
                yield (i, *val)

    parts = list(partitions(k))
    for t in unique_comb_rec(nb_vec - 1, 0):
        yield tuple(parts[ti] for ti in t)

    for t in unique_comb_rec(nb_vec - 2, 0):
        yield tuple(parts[ti] for ti in t) + ("I",)


print(list(unique_combinations(3, 3)))


def isotypic_tensor(*lams, n, k) -> np.ndarray:
    return tensor(*[np.identity(n**k) if pi_i == "I" else isotypic_projector(pi_i, n, k) for pi_i in lams])

[([3], [3], [3]), ([3], [3], [2, 1]), ([3], [3], [1, 1, 1]), ([3], [2, 1], [2, 1]), ([3], [2, 1], [1, 1, 1]), ([3], [1, 1, 1], [1, 1, 1]), ([2, 1], [2, 1], [2, 1]), ([2, 1], [2, 1], [1, 1, 1]), ([2, 1], [1, 1, 1], [1, 1, 1]), ([1, 1, 1], [1, 1, 1], [1, 1, 1]), ([3], [3], 'I'), ([3], [2, 1], 'I'), ([3], [1, 1, 1], 'I'), ([2, 1], [2, 1], 'I'), ([2, 1], [1, 1, 1], 'I'), ([1, 1, 1], [1, 1, 1], 'I')]


In [13]:
def check_zeros_on_isotypic_tensor(state:np.ndarray, n:int,k:int,sys:int):  
    to_zero = []
    non_zero = []

    for pis in tqdm(list(unique_combinations(3, 3))):
        pi_tensor = isotypic_tensor(*pis, n=n, k=k)
        if np.linalg.norm(pi_tensor @ state) < 1e-8:
            to_zero.append(pis)
        else:
            non_zero.append(pis)

    return to_zero, non_zero

In [14]:
n = 3
k = 3
sys = 3
state = random_state_vector(n**sys)
stateCopies = tensor([state] * k)
stateCopies = permute_systems(stateCopies, interleave_perm(k, sys), dim=[n] * k * sys)
to_zero, non_zero = check_zeros_on_isotypic_tensor(stateCopies, n, k, sys)
print("to zero")
print(to_zero)
print("\nnon_zero")
print(non_zero)

100%|██████████| 16/16 [00:46<00:00,  2.90s/it]

to zero
[([3], [3], [2, 1]), ([3], [3], [1, 1, 1]), ([3], [2, 1], [1, 1, 1]), ([2, 1], [1, 1, 1], [1, 1, 1]), ([1, 1, 1], [1, 1, 1], [1, 1, 1])]

non_zero
[([3], [3], [3]), ([3], [2, 1], [2, 1]), ([3], [1, 1, 1], [1, 1, 1]), ([2, 1], [2, 1], [2, 1]), ([2, 1], [2, 1], [1, 1, 1]), ([3], [3], 'I'), ([3], [2, 1], 'I'), ([3], [1, 1, 1], 'I'), ([2, 1], [2, 1], 'I'), ([2, 1], [1, 1, 1], 'I'), ([1, 1, 1], [1, 1, 1], 'I')]


In [15]:
n = 3
k = 3
sys = 3
for i in range(30):
    state = random_state_vector((n**sys) ** k)
    to_zero, non_zero = check_zeros_on_isotypic_tensor(state, n, k, sys)
    print("to zero")
    print(to_zero)
    # print("\nnon_zero")
    # print(non_zero)

100%|██████████| 16/16 [00:41<00:00,  2.62s/it]


to zero
[]


100%|██████████| 16/16 [00:42<00:00,  2.65s/it]


to zero
[]


100%|██████████| 16/16 [00:41<00:00,  2.62s/it]


...

to zero
[]


100%|██████████| 16/16 [00:43<00:00,  2.73s/it]


to zero
[]


  0%|          | 0/16 [00:02<?, ?it/s]


KeyboardInterrupt: 